# Chapter 23 Demo: Natural Language Processing

This notebook demonstrates the central ideas from Chapter 23, "Natural Language Processing":

- language models and smoothing
- bag-of-words representations
- part-of-speech tagging
- grammar and probabilistic context-free grammar
- bottom-up chart parsing
- augmented grammars with agreement features
- semantic interpretation
- ambiguity, scope, and pragmatics
- small examples of information extraction and question answering

The examples are intentionally small. They make the algorithms visible without requiring a large dataset or an internet connection.

## 0. Setup

The demo uses only Python standard-library tools. Optional table display uses `pandas` when it is installed, but every computation works without it.

In [1]:
import math
import random
import re
from collections import Counter, defaultdict
from itertools import product

random.seed(7)

# Thử dùng pandas để hiển thị bảng đẹp hơn; nếu máy không có pandas thì notebook vẫn chạy bình thường.
try:
    import pandas as pd
except Exception:
    pd = None


def tokenize(text):
    'Lowercase tokenizer for the toy examples.'
    # Biểu thức chính quy tách từ và dấu câu thành token riêng, đồng thời chuẩn hóa về chữ thường.
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[.!?]", text.lower())


def show_table(rows, columns=None):
    'Display a small table as a DataFrame when pandas is available.'
    # Nếu pandas có sẵn, DataFrame giúp kết quả dễ đọc trong notebook; nếu không thì trả về list gốc.
    if pd is not None:
        return pd.DataFrame(rows, columns=columns)
    return rows

## 1. Language Models

A language model assigns probabilities to word sequences. A small n-gram model estimates the next word from the previous `n - 1` words.

The key tradeoff is simple:

- larger contexts capture more local structure
- larger contexts are more likely to be sparse
- smoothing prevents unseen events from receiving probability zero

In [2]:
corpus = [
    "the wumpus is near .",
    "the pit is near .",
    "the gold is glittering .",
    "the agent grabs the gold .",
    "the agent smells the wumpus .",
    "the breeze means the pit is near .",
    "the stench means the wumpus is near .",
]

# Thêm <s> và </s> để mô hình học được vị trí bắt đầu/kết thúc câu.
tokenized = [["<s>"] + tokenize(sentence) + ["</s>"] for sentence in corpus]

# Set comprehension gom mọi token khác nhau; sorted giúp thứ tự từ vựng ổn định khi chạy lại.
vocab = sorted({word for sent in tokenized for word in sent})

print("Corpus size:", len(corpus), "sentences")
print("Vocabulary size:", len(vocab))
vocab

Corpus size: 7 sentences
Vocabulary size: 16


['.',
 '</s>',
 '<s>',
 'agent',
 'breeze',
 'glittering',
 'gold',
 'grabs',
 'is',
 'means',
 'near',
 'pit',
 'smells',
 'stench',
 'the',
 'wumpus']

In [3]:
def train_ngram(sentences, n):
    context_counts = Counter()
    ngram_counts = Counter()

    for sent in sentences:
        # Với n-gram, cần n-1 ký hiệu bắt đầu để câu có đủ ngữ cảnh ở token đầu tiên.
        padded = ["<s>"] * (n - 1) + [w for w in sent if w != "<s>"]
        for i in range(n - 1, len(padded)):
            # context là n-1 token đứng ngay trước từ đang xét.
            context = tuple(padded[i - n + 1:i])
            word = padded[i]
            context_counts[context] += 1
            ngram_counts[context + (word,)] += 1

    return {"n": n, "context_counts": context_counts, "ngram_counts": ngram_counts}


def next_word_probability(model, context, word, vocabulary, k=0.0):
    n = model["n"]
    # Chỉ giữ đúng độ dài ngữ cảnh mà mô hình n-gram cần.
    context = tuple(context[-(n - 1):]) if n > 1 else tuple()

    # Add-k smoothing cộng k vào tử số và phân bổ k * |V| vào mẫu số để tránh xác suất 0.
    numerator = model["ngram_counts"][context + (word,)] + k
    denominator = model["context_counts"][context] + k * len(vocabulary)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def sentence_log_probability(model, words, vocabulary, k=0.0):
    n = model["n"]
    padded = ["<s>"] * (n - 1) + tokenize(words) + ["</s>"]
    logp = 0.0
    details = []
    for i in range(n - 1, len(padded)):
        context = tuple(padded[i - n + 1:i])
        word = padded[i]
        p = next_word_probability(model, context, word, vocabulary, k=k)
        details.append((context, word, p))
        if p == 0:
            return float("-inf"), details

        # Dùng log probability để biến tích nhiều xác suất nhỏ thành tổng, ổn định số học hơn.
        logp += math.log(p)
    return logp, details


bigram = train_ngram(tokenized, n=2)
trigram = train_ngram(tokenized, n=3)

rows = []
for context in [("the",), ("wumpus",), ("gold",), ("pit",)]:
    # Sắp xếp toàn bộ từ vựng theo xác suất P(word | context), rồi lấy 5 ứng viên cao nhất.
    candidates = sorted(vocab, key=lambda w: next_word_probability(bigram, context, w, vocab, k=0.5), reverse=True)[:5]
    rows.append((" ".join(context), ", ".join(candidates)))

show_table(rows, ["Context", "Top next-word candidates with add-0.5 smoothing"])

,Context,Top next-word candidates with add-0.5 smoothing
0,the,"wumpus, agent, gold, pit, breeze"
1,wumpus,"is, ., </s>, <s>, agent"
2,gold,"., is, </s>, <s>, agent"
3,pit,"is, ., </s>, <s>, agent"


In [4]:
test_sentences = [
    "the wumpus is near .",
    "the gold is near .",
    "the agent is glittering .",
]

rows = []
for sent in test_sentences:
    raw_logp, _ = sentence_log_probability(bigram, sent, vocab, k=0.0)

    # So sánh với smoothing để thấy câu có n-gram chưa từng gặp vẫn nhận xác suất hữu hạn.
    smooth_logp, _ = sentence_log_probability(bigram, sent, vocab, k=0.5)
    rows.append((sent, raw_logp, smooth_logp))

show_table(rows, ["Sentence", "Bigram log P without smoothing", "Bigram log P with add-0.5"])

,Sentence,Bigram log P without smoothing,Bigram log P with add-0.5
0,the wumpus is near .,-1.927892,-6.601276
1,the gold is near .,-2.621039,-7.353264
2,the agent is glittering .,-inf,-10.361419


In [5]:
def generate_sentence(model, vocabulary, k=0.5, max_len=12):
    n = model["n"]
    context = ["<s>"] * (n - 1)
    words = []
    choices = [w for w in vocabulary if w != "<s>"]

    for _ in range(max_len):
        # Tạo trọng số lấy mẫu từ phân phối xác suất của từ kế tiếp trong ngữ cảnh hiện tại.
        weights = [next_word_probability(model, context, w, vocabulary, k=k) for w in choices]
        word = random.choices(choices, weights=weights, k=1)[0]
        if word == "</s>":
            break

        # Cập nhật ngữ cảnh bằng từ vừa sinh để bước sau dựa trên lịch sử mới nhất.
        words.append(word)
        context.append(word)
    return " ".join(words)


for i in range(5):
    print(f"{i + 1}.", generate_sentence(trigram, vocab, k=0.5))

1. near agent near
2. the glittering . grabs . grabs
3. agent grabs the agent glittering near wumpus means gold wumpus . stench
4. means agent
5. means stench agent means near gold means . . breeze pit grabs


### Character N-grams

Character models are useful for misspellings and unknown words because they can score partial word shapes instead of whole-word identities.

In [6]:
word_list = ["wumpus", "pit", "gold", "glittering", "breeze", "stench", "agent", "near"]


def char_ngrams(word, n=3):
    # Dùng ^ và $ để mô hình biết vị trí đầu/cuối của một từ.
    padded = "^" * (n - 1) + word + "$"
    return [padded[i:i+n] for i in range(len(padded) - n + 1)]


char_counts = Counter()
context_counts = Counter()
for word in word_list:
    padded = "^^" + word + "$"
    for i in range(2, len(padded)):
        # Context ở đây là 2 ký tự trước đó, vì ví dụ đang dùng character trigram.
        context = padded[i-2:i]
        char = padded[i]
        context_counts[context] += 1
        char_counts[(context, char)] += 1

alphabet = sorted(set("".join(word_list) + "$"))


def char_word_score(word, k=0.5):
    padded = "^^" + word + "$"
    score = 0.0
    for i in range(2, len(padded)):
        context = padded[i-2:i]
        char = padded[i]

        # Công thức smoothing giống word n-gram, nhưng đơn vị dự đoán là ký tự.
        p = (char_counts[(context, char)] + k) / (context_counts[context] + k * len(alphabet))
        score += math.log(p)
    return score


candidates = ["wumpus", "wumpuz", "wmpus", "gold", "goold", "stench"]
rows = [(word, char_word_score(word)) for word in candidates]
show_table(sorted(rows, key=lambda row: row[1], reverse=True), ["Candidate", "Character trigram log score"])

,Candidate,Character trigram log score
0,gold,-10.030813
1,goold,-14.125158
2,wumpus,-14.439488
3,wmpus,-14.548972
4,stench,-14.613510
5,wumpuz,-16.541402


## 2. Bag-of-Words Models

A bag-of-words model represents a document by word counts or word presence. It is often strong for topic or sentiment clues, but it ignores syntax and word order.

In [7]:
def bow_vector(text):
    # Counter biến văn bản thành vector đếm từ, bỏ qua thứ tự xuất hiện của từ.
    return Counter(w for w in tokenize(text) if re.match(r"[a-z]", w))


sentence_a = "dog bites man"
sentence_b = "man bites dog"

rows = []
for word in sorted(set(bow_vector(sentence_a)) | set(bow_vector(sentence_b))):
    rows.append((word, bow_vector(sentence_a)[word], bow_vector(sentence_b)[word]))

print("The two sentences have the same bag-of-words vector, but different meanings.")
show_table(rows, ["Word", "dog bites man", "man bites dog"])

The two sentences have the same bag-of-words vector, but different meanings.


,Word,dog bites man,man bites dog
0,bites,1,1
1,dog,1,1
2,man,1,1


In [8]:
train_docs = [
    ("good fun bright", "positive"),
    ("good helpful clear", "positive"),
    ("excellent fun useful", "positive"),
    ("bad boring dull", "negative"),
    ("bad confusing slow", "negative"),
    ("dull boring useless", "negative"),
]


def train_multinomial_nb(examples):
    labels = sorted({label for _, label in examples})
    label_counts = Counter(label for _, label in examples)
    word_counts = {label: Counter() for label in labels}
    total_words = Counter()
    vocab = set()

    for text, label in examples:
        words = [w for w in tokenize(text) if re.match(r"[a-z]", w)]

        # Lưu số lần mỗi từ xuất hiện theo từng nhãn để ước lượng P(word | label).
        word_counts[label].update(words)
        total_words[label] += len(words)
        vocab.update(words)

    return labels, label_counts, word_counts, total_words, sorted(vocab)


def nb_predict(model, text, alpha=1.0):
    labels, label_counts, word_counts, total_words, vocab = model
    words = [w for w in tokenize(text) if re.match(r"[a-z]", w)]
    scores = {}
    for label in labels:
        # Bắt đầu bằng log prior P(label).
        score = math.log(label_counts[label] / sum(label_counts.values()))
        for word in words:
            # Cộng log likelihood với Laplace smoothing để từ hiếm/không thấy không làm xác suất bằng 0.
            score += math.log((word_counts[label][word] + alpha) / (total_words[label] + alpha * len(vocab)))
        scores[label] = score
    return max(scores, key=scores.get), scores


nb_model = train_multinomial_nb(train_docs)

tests = ["good useful fun", "bad dull slow", "not good", "not bad"]
rows = []
for text in tests:
    label, scores = nb_predict(nb_model, text)
    rows.append((text, label, round(scores["positive"], 3), round(scores["negative"], 3)))

show_table(rows, ["Text", "Predicted label", "Log score positive", "Log score negative"])

,Text,Predicted label,Log score positive,Log score negative
0,good useful fun,positive,-7.076,-9.966
1,bad dull slow,negative,-9.966,-7.076
2,not good,positive,-5.777,-6.875
3,not bad,negative,-6.875,-5.777


The bag-of-words classifier treats `not good` as containing the positive clue `good`. This is a small example of why word order, grammar, and context matter.

## 3. Part-of-Speech Tagging

Part-of-speech tags label words as nouns, verbs, determiners, prepositions, and so on. The same word can have different tags, so context is essential.

The next cell uses Viterbi decoding for a tiny hidden Markov model.

In [9]:
states = ["N", "V", "DET", "P"]

start_prob = {"N": 0.35, "V": 0.10, "DET": 0.45, "P": 0.10}

transition_prob = {
    "N": {"N": 0.05, "V": 0.55, "DET": 0.05, "P": 0.20, "END": 0.15},
    "V": {"N": 0.20, "V": 0.05, "DET": 0.35, "P": 0.30, "END": 0.10},
    "DET": {"N": 0.90, "V": 0.02, "DET": 0.02, "P": 0.03, "END": 0.03},
    "P": {"N": 0.35, "V": 0.03, "DET": 0.57, "P": 0.02, "END": 0.03},
}

emission_prob = {
    "N": {"time": 0.30, "flies": 0.20, "arrow": 0.30, "fruit": 0.20},
    "V": {"time": 0.15, "flies": 0.30, "like": 0.30, "saw": 0.25},
    "DET": {"an": 0.50, "the": 0.50},
    "P": {"like": 0.80, "with": 0.20},
}


def log(x):
    return math.log(x) if x > 0 else float("-inf")


def viterbi(words):
    table = []
    back = []

    first = {}
    first_back = {}
    for state in states:
        emission = emission_prob[state].get(words[0], 0.001)

        # Điểm ban đầu = xác suất bắt đầu bằng tag đó * xác suất tag sinh ra từ đầu tiên.
        first[state] = log(start_prob[state]) + log(emission)
        first_back[state] = None
    table.append(first)
    back.append(first_back)

    for i in range(1, len(words)):
        row = {}
        row_back = {}
        for state in states:
            emission = emission_prob[state].get(words[i], 0.001)
            candidates = []
            for prev in states:
                # Viterbi chỉ giữ đường đi tốt nhất đến mỗi state, thay vì liệt kê mọi chuỗi tag.
                candidates.append((table[i - 1][prev] + log(transition_prob[prev][state]) + log(emission), prev))
            row[state], row_back[state] = max(candidates)
        table.append(row)
        back.append(row_back)

    final_candidates = [(table[-1][state] + log(transition_prob[state]["END"]), state) for state in states]
    best_score, best_state = max(final_candidates)

    # Truy vết ngược qua backpointer để khôi phục chuỗi tag tốt nhất.
    tags = [best_state]
    for i in range(len(words) - 1, 0, -1):
        tags.append(back[i][tags[-1]])
    tags.reverse()
    return list(zip(words, tags)), best_score


sentence = "time flies like an arrow"
tagged, score = viterbi(tokenize(sentence))
print("Sentence:", sentence)
print("Best tag sequence:", tagged)
print("Log score:", round(score, 3))

Sentence: time flies like an arrow
Best tag sequence: [('time', 'N'), ('flies', 'V'), ('like', 'P'), ('an', 'DET'), ('arrow', 'N')]
Log score: -9.944


## 4. Grammar and PCFG

A context-free grammar defines how phrases can be built. A probabilistic context-free grammar, or PCFG, attaches a probability to each rule.

The probability of a parse tree is the product of the probabilities of its rules. With logs, this becomes a sum of log probabilities.

In [10]:
pcfg_rules = [
    ("S",  ("NP", "VP"), 1.00),
    ("NP", ("Det", "N"), 0.45),
    ("NP", ("NP", "PP"), 0.25),
    ("NP", ("i",), 0.30),
    ("VP", ("V", "NP"), 0.60),
    ("VP", ("VP", "PP"), 0.40),
    ("PP", ("P", "NP"), 1.00),
    ("Det", ("the",), 1.00),
    ("N", ("wumpus",), 0.35),
    ("N", ("telescope",), 0.25),
    ("N", ("pit",), 0.20),
    ("N", ("gold",), 0.20),
    ("V", ("saw",), 0.70),
    ("V", ("smelled",), 0.30),
    ("P", ("with",), 0.60),
    ("P", ("near",), 0.40),
]


def split_rules(rules):
    # Tập nonterminal giúp phân biệt luật từ vựng A -> word và luật cú pháp A -> B C.
    nonterminals = {lhs for lhs, _, _ in rules}
    lexical = []
    binary = []
    for lhs, rhs, prob in rules:
        if len(rhs) == 1 and rhs[0] not in nonterminals:
            lexical.append((lhs, rhs[0], math.log(prob)))
        elif len(rhs) == 2:
            binary.append((lhs, rhs[0], rhs[1], math.log(prob)))
        else:
            raise ValueError(f"Unsupported rule: {lhs} -> {rhs}")
    return lexical, binary


lexical_rules, binary_rules = split_rules(pcfg_rules)

show_table(
    [(lhs, " ".join(rhs), prob) for lhs, rhs, prob in pcfg_rules],
    ["Left side", "Right side", "Rule probability"],
)

,Left side,Right side,Rule probability
0,S,NP VP,1.00
1,NP,Det N,0.45
2,NP,NP PP,0.25
3,NP,i,0.30
4,VP,V NP,0.60
5,VP,VP PP,0.40
6,PP,P NP,1.00
7,Det,the,1.00
8,N,wumpus,0.35
9,N,telescope,0.25


## 5. Parsing as Search: A Bottom-Up Chart Parser

Parsing finds a phrase structure for a string. A chart parser stores partial parses by span, so a subphrase is computed once and reused.

The parser below is CYK-like: it fills spans from short to long and combines constituents with binary grammar rules.

In [11]:
def tree_to_string(tree, indent=0):
    label, children = tree
    pad = "  " * indent
    if isinstance(children, str):
        return f"{pad}({label} {children})"
    lines = [f"{pad}({label}"]
    for child in children:
        lines.append(tree_to_string(child, indent + 1))
    lines[-1] += ")"
    return "\n".join(lines)


def chart_parse(words, keep_per_symbol=5):
    n = len(words)

    # chart[i][j] lưu các cây parse tốt nhất cho đoạn words[i:j].
    chart = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n)]

    for i, word in enumerate(words):
        for lhs, terminal, logprob in lexical_rules:
            if terminal == word:
                chart[i][i + 1][lhs].append((logprob, (lhs, word)))

    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                # Thử mọi điểm tách k để ghép constituent bên trái và bên phải.
                for lhs, left, right, rule_logprob in binary_rules:
                    for left_logprob, left_tree in chart[i][k].get(left, []):
                        for right_logprob, right_tree in chart[k][j].get(right, []):
                            total = rule_logprob + left_logprob + right_logprob
                            chart[i][j][lhs].append((total, (lhs, [left_tree, right_tree])))

            for symbol in list(chart[i][j].keys()):
                # Giữ lại một số cây tốt nhất cho mỗi nonterminal để bảng không phình quá lớn.
                chart[i][j][symbol].sort(key=lambda item: item[0], reverse=True)
                chart[i][j][symbol] = chart[i][j][symbol][:keep_per_symbol]

    return chart


sentence = "i saw the wumpus with the telescope"
words = tokenize(sentence)
chart = chart_parse(words)
parses = chart[0][len(words)]["S"]

print("Sentence:", sentence)
print("Number of S parses kept:", len(parses))
for rank, (logprob, tree) in enumerate(parses, 1):
    print("\nParse", rank, "log probability:", round(logprob, 3))
    print(tree_to_string(tree))

Sentence: i saw the wumpus with the telescope
Number of S parses kept: 2

Parse 1 log probability: -7.532
(S
  (NP i)
  (VP
    (VP
      (V saw)
      (NP
        (Det the)
        (N wumpus)))
    (PP
      (P with)
      (NP
        (Det the)
        (N telescope)))))

Parse 2 log probability: -8.002
(S
  (NP i)
  (VP
    (V saw)
    (NP
      (NP
        (Det the)
        (N wumpus))
      (PP
        (P with)
        (NP
          (Det the)
          (N telescope))))))


In [12]:
def chart_summary(chart, words):
    rows = []
    n = len(words)
    for span in range(1, n + 1):
        for i in range(n - span + 1):
            j = i + span
            symbols = sorted(chart[i][j].keys())
            if symbols:
                # Mỗi dòng tóm tắt các nonterminal có thể tạo ra một span cụ thể.
                rows.append((i, j, " ".join(words[i:j]), ", ".join(symbols)))
    return rows


show_table(chart_summary(chart, words), ["Start", "End", "Words in span", "Constituents"])

,Start,End,Words in span,Constituents
0,0,1,i,NP
1,1,2,saw,V
2,2,3,the,Det
3,3,4,wumpus,N
4,4,5,with,P
5,5,6,the,Det
6,6,7,telescope,N
7,2,4,the wumpus,NP
8,5,7,the telescope,NP
9,1,4,saw the wumpus,VP


The sentence has two plausible PP attachments:

- `with the telescope` modifies the act of seeing
- `with the telescope` modifies the wumpus

The PCFG ranks these structures by rule probabilities, but real NLP systems often need semantics and world knowledge too.

## 6. Augmented Grammars

Plain CFG categories such as `NP` and `VP` are often too coarse. Augmented grammars attach features such as person, number, case, tense, and head words.

This small feature checker enforces subject-verb agreement.

In [13]:
pronoun_features = {
    "i": {"category": "NP", "person_number": "1sg"},
    "you": {"category": "NP", "person_number": "2"},
    "we": {"category": "NP", "person_number": "pl"},
    "they": {"category": "NP", "person_number": "pl"},
    "she": {"category": "NP", "person_number": "3sg"},
    "he": {"category": "NP", "person_number": "3sg"},
}

verb_features = {
    "eat": {"category": "V", "allowed_subjects": {"1sg", "2", "pl"}},
    "eats": {"category": "V", "allowed_subjects": {"3sg"}},
    "see": {"category": "V", "allowed_subjects": {"1sg", "2", "pl"}},
    "sees": {"category": "V", "allowed_subjects": {"3sg"}},
}

nouns = {"bananas", "gold", "wumpus", "pit"}


def agreement_parse(sentence):
    words = tokenize(sentence)
    if len(words) != 3:
        return False, "Expected: pronoun verb object"

    subj, verb, obj = words
    if subj not in pronoun_features:
        return False, "Unknown subject"
    if verb not in verb_features:
        return False, "Unknown verb"
    if obj not in nouns:
        return False, "Unknown object"

    # So khớp đặc trưng person-number của chủ ngữ với tập chủ ngữ mà động từ cho phép.
    subject_pn = pronoun_features[subj]["person_number"]
    allowed = verb_features[verb]["allowed_subjects"]
    if subject_pn not in allowed:
        return False, f"Agreement failure: subject is {subject_pn}, but verb '{verb}' allows {sorted(allowed)}"

    return True, f"Valid: NP({subject_pn}) + VP({verb})"


tests = ["i eat bananas", "she eats bananas", "she eat bananas", "they sees gold"]
rows = [(sent, *agreement_parse(sent)) for sent in tests]
show_table(rows, ["Sentence", "Accepted?", "Explanation"])

,Sentence,Accepted?,Explanation
0,i eat bananas,True,Valid: NP(1sg) + VP(eat)
1,she eats bananas,True,Valid: NP(3sg) + VP(eats)
2,she eat bananas,False,"Agreement failure: subject is 3sg, but verb 'e..."
3,they sees gold,False,"Agreement failure: subject is pl, but verb 'se..."


### Lexicalized Preferences

A lexicalized PCFG can condition probabilities on head words. The next tiny example gives different scores to verb-object pairs.

In [14]:
verb_object_preference = {
    ("eat", "banana"): 0.80,
    ("eat", "bandanna"): 0.05,
    ("wear", "banana"): 0.05,
    ("wear", "bandanna"): 0.75,
}

# Ví dụ lexicalized preference: xác suất phụ thuộc vào cặp head verb và object, không chỉ vào nhãn cú pháp.
for verb, obj in [("eat", "banana"), ("eat", "bandanna"), ("wear", "banana"), ("wear", "bandanna")]:
    print(f"P(object={obj!r} | verb={verb!r}) = {verb_object_preference[(verb, obj)]}")

P(object='banana' | verb='eat') = 0.8
P(object='bandanna' | verb='eat') = 0.05
P(object='banana' | verb='wear') = 0.05
P(object='bandanna' | verb='wear') = 0.75


## 7. Semantic Interpretation

Syntax explains how a sentence is built. Semantics connects that structure to meaning.

The following toy interpreter composes a logical form for simple subject-verb-object sentences.

In [15]:
names = {"ali": "Ali", "bo": "Bo", "cy": "Cy"}


def love_relation(obj):
    # Hàm trả về một hàm khác: nhận object trước, rồi chờ subject để tạo logical form.
    return lambda subj: f"Loves({subj}, {obj})"


def see_relation(obj):
    return lambda subj: f"Sees({subj}, {obj})"


verb_meanings = {
    "loves": love_relation,
    "sees": see_relation,
}


def interpret_svo(sentence):
    words = tokenize(sentence)
    if len(words) != 3:
        raise ValueError("Expected exactly: Name Verb Name")
    subj_word, verb_word, obj_word = words
    subj = names[subj_word]
    obj = names[obj_word]

    # Áp dụng nghĩa của động từ theo kiểu compositional semantics: verb(object)(subject).
    return verb_meanings[verb_word](obj)(subj)


for sentence in ["ali loves bo", "bo loves ali", "cy sees bo"]:
    print(sentence, "=>", interpret_svo(sentence))

ali loves bo => Loves(Ali, Bo)
bo loves ali => Loves(Bo, Ali)
cy sees bo => Sees(Cy, Bo)


In [16]:
facts = {
    "Loves(Ali, Bo)",
    "Sees(Cy, Bo)",
}

for question in ["ali loves bo", "bo loves ali", "cy sees bo"]:
    logical_form = interpret_svo(question)

    # Sau khi câu được đổi thành logical form, việc kiểm tra đúng/sai chỉ là tra trong tập facts.
    print(f"Is it true that '{question}'?", logical_form in facts, "|", logical_form)

Is it true that 'ali loves bo'? True | Loves(Ali, Bo)
Is it true that 'bo loves ali'? False | Loves(Bo, Ali)
Is it true that 'cy sees bo'? True | Sees(Cy, Bo)


## 8. Complications of Real Natural Language

Real language adds scope, pragmatics, tense, ellipsis, long-distance dependencies, ambiguity, and metonymy.

The next example shows quantifier scope. The sentence is:

`Every robot saw a pit.`

Two readings are possible:

- narrow-scope existential: every robot saw at least one pit, possibly different pits
- wide-scope existential: there is one pit that every robot saw

In [17]:
robots = {"r1", "r2"}
world_a = {
    ("r1", "p1"),
    ("r2", "p2"),
}
world_b = {
    ("r1", "p1"),
    ("r2", "p1"),
}
pits = {"p1", "p2"}


def every_robot_saw_a_pit_narrow(world):
    # Narrow scope: với mỗi robot, chỉ cần tồn tại ít nhất một pit mà robot đó thấy.
    return all(any((robot, pit) in world for pit in pits) for robot in robots)


def every_robot_saw_a_pit_wide(world):
    # Wide scope: phải có cùng một pit được tất cả robot nhìn thấy.
    return any(all((robot, pit) in world for robot in robots) for pit in pits)


rows = []
for name, world in [("World A: different pits", world_a), ("World B: shared pit", world_b)]:
    rows.append((name, every_robot_saw_a_pit_narrow(world), every_robot_saw_a_pit_wide(world)))

show_table(rows, ["World", "Narrow-scope reading", "Wide-scope reading"])

,World,Narrow-scope reading,Wide-scope reading
0,World A: different pits,True,False
1,World B: shared pit,True,True


Pragmatics can change how an utterance should be interpreted. For example, `Can you pass the gold?` is grammatically a yes/no question, but in context it often functions as a request.

In [18]:
def simple_speech_act(utterance):
    text = utterance.lower().strip()

    # Quy tắc heuristic: câu bắt đầu bằng "can you/could you" thường là lời yêu cầu, không chỉ là câu hỏi.
    if text.startswith("can you ") or text.startswith("could you "):
        return "request"
    if text.endswith("?"):
        return "question"
    if text.startswith("please "):
        return "request"
    if text.endswith("!"):
        return "command or exclamation"
    return "statement"


for utterance in [
    "Can you pass the gold?",
    "Is the pit near?",
    "Please move east.",
    "The wumpus is near.",
]:
    print(f"{utterance!r} -> {simple_speech_act(utterance)}")

'Can you pass the gold?' -> request
'Is the pit near?' -> question
'Please move east.' -> request
'The wumpus is near.' -> statement


## 9. Natural Language Tasks: Information Extraction and QA

Many NLP applications transform text into structured data. A simple information-extraction system can find entities, relations, events, dates, and locations.

The next example extracts small Wumpus-world facts and answers questions from them.

In [19]:
documents = [
    "Alice grabbed the gold in room1.",
    "Bob smelled the wumpus in room2.",
    "Alice saw a pit in room3.",
    "The breeze means a pit is near room3.",
]

patterns = [
    # Named groups (?P<name>...) giúp lấy trực tiếp agent/action/object/location từ câu.
    ("event", re.compile(r"(?P<agent>[A-Z][a-z]+) (?P<action>grabbed|smelled|saw) (?:the|a) (?P<object>[a-z]+) in (?P<location>room[0-9]+)", re.I)),
    ("clue", re.compile(r"the (?P<clue>breeze|stench) means (?:the|a) (?P<object>[a-z]+) is near (?P<location>room[0-9]+)", re.I)),
]


def extract_facts(docs):
    facts = []
    for doc in docs:
        for kind, pattern in patterns:
            match = pattern.search(doc)
            if match:
                # groupdict() chuyển các named groups thành dictionary; ** dùng để trộn vào fact mới.
                item = {"kind": kind, **{k: v.lower() for k, v in match.groupdict().items()}}
                facts.append(item)
                break
    return facts


facts = extract_facts(documents)
show_table(facts)

,kind,agent,action,object,location,clue
0,event,alice,grabbed,gold,room1,NaN
1,event,bob,smelled,wumpus,room2,NaN
2,event,alice,saw,pit,room3,NaN
3,clue,NaN,NaN,pit,room3,breeze


In [20]:
def answer_question(question, facts):
    q = question.lower()

    m = re.match(r"what did ([a-z]+) (grab|smell|see)\?", q)
    if m:
        agent, action = m.groups()
        action_map = {"grab": "grabbed", "smell": "smelled", "see": "saw"}
        wanted = action_map[action]

        # List comprehension lọc các fact khớp agent và action, rồi lấy object làm câu trả lời.
        answers = [f["object"] for f in facts if f.get("agent") == agent and f.get("action") == wanted]
        return ", ".join(answers) if answers else "I do not know."

    m = re.match(r"where did ([a-z]+) (grab|smell|see) (?:the|a) ([a-z]+)\?", q)
    if m:
        agent, action, obj = m.groups()
        action_map = {"grab": "grabbed", "smell": "smelled", "see": "saw"}
        wanted = action_map[action]
        answers = [
            f["location"]
            for f in facts
            if f.get("agent") == agent and f.get("action") == wanted and f.get("object") == obj
        ]
        return ", ".join(answers) if answers else "I do not know."

    m = re.match(r"what is near (room[0-9]+)\?", q)
    if m:
        location = m.group(1)
        answers = [f["object"] for f in facts if f["kind"] == "clue" and f["location"] == location]
        return ", ".join(answers) if answers else "I do not know."

    return "I do not understand the question pattern."


questions = [
    "What did Alice grab?",
    "Where did Bob smell the wumpus?",
    "What is near room3?",
    "What did Bob grab?",
]

for question in questions:
    print(question, "->", answer_question(question, facts))

What did Alice grab? -> gold
Where did Bob smell the wumpus? -> room2
What is near room3? -> pit
What did Bob grab? -> I do not know.


## 10. Summary

This demo showed how Chapter 23 ideas connect:

- n-gram language models estimate likely word sequences but need smoothing
- bag-of-words models are useful but discard order and syntax
- POS tagging uses context to resolve word-category ambiguity
- PCFGs rank parse trees by rule probabilities
- chart parsing stores partial structures and avoids repeated work
- augmented grammars add features such as agreement and lexical heads
- semantic interpretation maps parsed language into logical forms or executable actions
- real language requires scope resolution, pragmatics, and world knowledge
- NLP tasks combine these tools to extract facts, translate, recognize speech, answer questions, and generate language

Suggested experiments:

1. Add more Wumpus-world sentences to the corpus and observe the n-gram predictions.
2. Change the PCFG probabilities and see how the preferred parse changes.
3. Add plural nouns and verbs to the agreement checker.
4. Add a new question pattern to the QA system.